In [ ]:
import json
import pandas as pd
import numpy as np
import os
from chronos import Chronos2Pipeline
from sklearn.metrics import root_mean_squared_error

# ============================================================
# CONFIG
# ============================================================
DAYS_JSON = r"C:\Users\CR58XM\Documents\GitHub\AAU_learning_to_predict_together_or_alone\dataset_days.json"
DATA_DIR  = r"C:\Users\CR58XM\Documents\GitHub\AAU_learning_to_predict_together_or_alone\DataCleaning\clean"
OUT_DIR   = r"C:\Users\CR58XM\Documents\GitHub\AAU_learning_to_predict_together_or_alone\Outputs"
MODEL_DIR = r"C:\Users\CR58XM\Documents\GitHub\AAU_learning_to_predict_together_or_alone\Models"

countries = ["Germany", "Ireland", "Portugal"]
days = ["day1", "day2", "day3", "day4", "day5"]

features = [
    "temperature_2m",
    "relative_humidity_2m",
    "wind_speed_10m",
    "precipitation",
    "direct_radiation",
]

PRED_LEN = 96
TAIL_TRAIN_PREDICT = 10000   # context length used in predict_df input
TAIL_TRAIN_FINETUNE = 20000  # context length used to build finetune inputs

# Your baseline CV RMSE (from zero-shot run) so we can compute improvement %
baseline_cv_rmse = {
    "Germany": 680.116128,
    "Ireland": 535.503877,
    "Portugal": 287.043478,
}

# Fine-tuning hyperparameters (LoRA)
FT_MODE = "lora"
FT_NUM_STEPS = 1000
FT_LR = 1e-5
FT_BATCH_SIZE = 32
FT_LOGGING_STEPS = 100

# ============================================================
# LOAD DAY CUTOFFS
# ============================================================
with open(DAYS_JSON, "r") as f:
    dataset_days = json.load(f)

os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

# ============================================================
# HELPERS
# ============================================================
def build_finetune_inputs(df: pd.DataFrame,
                          households: list[str],
                          features: list[str],
                          cutoff: pd.Timestamp,
                          tail_n: int | None = 20000):
    """
    Build Chronos fit() inputs list-of-dicts for many household series with past covariates.
    Uses only rows with timestamp < cutoff.
    """
    df_ft = df.loc[df.index < cutoff].copy()
    train_inputs = []

    for h in households:
        s = df_ft[h].dropna()
        if len(s) < 512:
            continue

        X = df_ft.loc[s.index, features].copy()

        # Be robust to NaNs in covariates
        X = X.ffill().bfill()

        # keep last tail_n points for compute control
        if tail_n is not None:
            s = s.tail(tail_n)
            X = X.tail(len(s))

        train_inputs.append({
            "target": s.to_numpy(),
            "past_covariates": {col: X[col].to_numpy() for col in features},
            "future_covariates": {},  # no future covariates available
        })

    return train_inputs


def evaluate_pipeline_cv(pipeline: Chronos2Pipeline,
                         df: pd.DataFrame,
                         households: list[str],
                         features: list[str],
                         country: str,
                         days: list[str],
                         dataset_days: dict,
                         prediction_length: int = 96,
                         tail_train: int = 10000,
                         save_predictions: bool = True):
    """
    Rolling-origin evaluation:
    - For each day cutoff: train context is df < cutoff
    - Predict 96 steps
    - RMSE averaged across households for that day
    Optionally saves predictions per (country, day).
    """
    results = []

    for day in days:
        cutoff = pd.to_datetime(dataset_days[country][day])

        predictions_df_all_households = None
        rmse_households = []

        for h in households:
            s_train = df.loc[df.index < cutoff, h]
            X_train = df.loc[s_train.index, features].copy().ffill().bfill()

            df_household_train = pd.concat([s_train.rename(h), X_train], axis=1)
            df_household_train_last = df_household_train.tail(tail_train).reset_index()
            df_household_train_last["id"] = h

            pred_df = pipeline.predict_df(
                df_household_train_last,
                prediction_length=prediction_length,
                id_column="id",
                timestamp_column="timestamp",
                target=h,
            )

            # Initialize output df with the prediction timestamps from first household
            if predictions_df_all_households is None:
                if "timestamp" in pred_df.columns:
                    pred_index = pred_df["timestamp"]
                else:
                    # fallback if timestamp is returned as index
                    pred_index = pred_df.index
                predictions_df_all_households = pd.DataFrame(index=pred_index)

            predictions_df_all_households[h] = pred_df["predictions"].to_numpy()

            # RMSE for this household, aligned by timestamp
            y_pred = predictions_df_all_households[h]
            y_true = df.loc[y_pred.index, h]

            mask = (~pd.isna(y_true)) & (~pd.isna(y_pred))
            if mask.sum() == 0:
                continue

            rmse = root_mean_squared_error(y_true[mask], y_pred[mask])
            rmse_households.append(rmse)

        avg_rmse = float(np.mean(rmse_households)) if rmse_households else np.nan
        results.append({"country": country, "day": day, "rmse": avg_rmse})

        # Save predictions for this day (optional)
        #if save_predictions and predictions_df_all_households is not None:
        #    out_path = os.path.join(OUT_DIR, f"Chronos2Univar_pred_{day}_{country.capitalize()}.csv")
        #    predictions_df_all_households.to_csv(out_path, index=True)
        #    print("      Saved predictions:", out_path)

    rmse_df = pd.DataFrame(results)
    cv_mean = float(rmse_df["rmse"].mean())
    return rmse_df, cv_mean


# ============================================================
# FINE-TUNE + EVALUATE PER COUNTRY
# ============================================================
finetune_summary = []

for country in countries:
    print("\n==============================")
    print("Country:", country)

    # Load dataset once per country
    data_path = os.path.join(DATA_DIR, f"dataset_{country.capitalize()}.csv")
    df = pd.read_csv(data_path, index_col="timestamp", parse_dates=True).sort_index()

    households = [col for col in df.columns if col not in features]

    # Re-initialize base pipeline per country (prevents any accidental carryover)
    base_pipeline = Chronos2Pipeline.from_pretrained("amazon/chronos-2", device_map="cuda")

    # Fine-tune cutoff: use ONLY data before day1 cutoff for fine-tuning
    ft_cutoff = pd.to_datetime(dataset_days[country]["day1"])

    train_inputs = build_finetune_inputs(
        df=df,
        households=households,
        features=features,
        cutoff=ft_cutoff,
        tail_n=TAIL_TRAIN_FINETUNE
    )
    print("Fine-tune series count:", len(train_inputs))

    # ---- LoRA fine-tune ----
    finetuned_pipeline = base_pipeline.fit(
        inputs=train_inputs,
        finetune_mode=FT_MODE,            # "lora" or "full"
        prediction_length=PRED_LEN,
        num_steps=FT_NUM_STEPS,
        learning_rate=FT_LR,
        batch_size=FT_BATCH_SIZE,
        logging_steps=FT_LOGGING_STEPS,
    )

    # ---- SAVE MODEL (robust) ----
    model_dir = os.path.join(MODEL_DIR, f"Chronos2_{FT_MODE.upper()}_{country}")
    os.makedirs(model_dir, exist_ok=True)

    # Best: save the full pipeline if supported
    if hasattr(finetuned_pipeline, "save_pretrained"):
        finetuned_pipeline.save_pretrained(model_dir)
    else:
        # Fallback: save model weights (may be enough depending on Chronos version)
        finetuned_pipeline.model.save_pretrained(model_dir)

    # Always save metadata
    meta = {
        "country": country,
        "prediction_length": PRED_LEN,
        "features": features,
        "finetune_mode": FT_MODE,
        "num_steps": FT_NUM_STEPS,
        "learning_rate": FT_LR,
        "batch_size": FT_BATCH_SIZE,
        "tail_train_finetune": TAIL_TRAIN_FINETUNE,
        "tail_train_predict": TAIL_TRAIN_PREDICT,
        "finetune_cutoff_day": "day1",
        "finetune_cutoff_timestamp": str(ft_cutoff),
    }
    with open(os.path.join(model_dir, "meta.json"), "w") as f:
        json.dump(meta, f, indent=2)

    print("Saved fine-tuned model to:", model_dir)

    # ---- EVALUATE (rolling-origin CV across all days) ----
    ft_rmse_df, ft_cv_mean = evaluate_pipeline_cv(
        pipeline=finetuned_pipeline,
        df=df,
        households=households,
        features=features,
        country=country,
        days=days,
        dataset_days=dataset_days,
        prediction_length=PRED_LEN,
        tail_train=TAIL_TRAIN_PREDICT,
        save_predictions=True,  # set False if you don't want to save CSVs during CV
    )

    base = baseline_cv_rmse.get(country, np.nan)
    improvement_pct = (base - ft_cv_mean) / base * 100.0 if np.isfinite(base) else np.nan

    finetune_summary.append({
        "country": country,
        "baseline_cv_rmse": base,
        "finetuned_cv_rmse": ft_cv_mean,
        "improvement_%": improvement_pct,
        "model_dir": model_dir
    })

    print("\nFine-tuned per-day RMSE:")
    print(ft_rmse_df)

    print("\nBaseline CV mean RMSE:", base)
    print("Fine-tuned CV mean RMSE:", ft_cv_mean)
    print("Improvement (%):", improvement_pct)

summary_df = pd.DataFrame(finetune_summary)
print("\n==============================")
print("Fine-tuning summary:")
print(summary_df)

# Optionally save summary table
summary_path = os.path.join(OUT_DIR, "Chronos2_finetune_summary.csv")
summary_df.to_csv(summary_path, index=False)
print("\nSaved summary to:", summary_path)


Country: Germany
Fine-tune series count: 28


Could not estimate the number of tokens of the input, floating-point operations will not be computed


Step,Training Loss


Saved fine-tuned model to: C:\Users\CR58XM\Documents\GitHub\AAU_learning_to_predict_together_or_alone\Models\Chronos2_LORA_Germany

Fine-tuned per-day RMSE:
   country   day         rmse
0  Germany  day1  1147.013705
1  Germany  day2   233.082159
2  Germany  day3   652.985211
3  Germany  day4  1121.036780
4  Germany  day5   251.196695

Baseline CV mean RMSE: 680.116128
Fine-tuned CV mean RMSE: 681.0629098016743
Improvement (%): -0.13920884429817837

Country: Ireland
Fine-tune series count: 20


Could not estimate the number of tokens of the input, floating-point operations will not be computed


Step,Training Loss


Saved fine-tuned model to: C:\Users\CR58XM\Documents\GitHub\AAU_learning_to_predict_together_or_alone\Models\Chronos2_LORA_Ireland

Fine-tuned per-day RMSE:
   country   day        rmse
0  Ireland  day1  767.099903
1  Ireland  day2  209.794601
2  Ireland  day3  546.091403
3  Ireland  day4  730.617501
4  Ireland  day5  413.219258

Baseline CV mean RMSE: 535.503877
Fine-tuned CV mean RMSE: 533.3645335570047
Improvement (%): 0.39950101855103

Country: Portugal
Fine-tune series count: 23


Could not estimate the number of tokens of the input, floating-point operations will not be computed


Step,Training Loss


Saved fine-tuned model to: C:\Users\CR58XM\Documents\GitHub\AAU_learning_to_predict_together_or_alone\Models\Chronos2_LORA_Portugal

Fine-tuned per-day RMSE:
    country   day        rmse
0  Portugal  day1  387.493472
1  Portugal  day2  221.274859
2  Portugal  day3  294.622106
3  Portugal  day4  350.692103
4  Portugal  day5  188.802791

Baseline CV mean RMSE: 287.043478
Fine-tuned CV mean RMSE: 288.5770661932624
Improvement (%): -0.5342703495469782

Fine-tuning summary:
    country  baseline_cv_rmse  finetuned_cv_rmse  improvement_%  \
0   Germany        680.116128         681.062910      -0.139209   
1   Ireland        535.503877         533.364534       0.399501   
2  Portugal        287.043478         288.577066      -0.534270   

                                           model_dir  
0  C:\Users\CR58XM\Documents\GitHub\AAU_learning_...  
1  C:\Users\CR58XM\Documents\GitHub\AAU_learning_...  
2  C:\Users\CR58XM\Documents\GitHub\AAU_learning_...  

Saved summary to: C:\Users\CR58XM\D

# make predictions after finetunning

In [ ]:
import json
import pandas as pd
import numpy as np
import os
from chronos import Chronos2Pipeline

# ============================================================
# PREDICT AFTER FINETUNING (BASE vs FINETUNED) FOR ALL COUNTRIES & DAYS
# Run this cell AFTER your fine-tune+save cell (models saved on disk).
# ============================================================

DAYS_JSON = r"C:\Users\CR58XM\Documents\GitHub\AAU_learning_to_predict_together_or_alone\dataset_days.json"
DATA_DIR  = r"C:\Users\CR58XM\Documents\GitHub\AAU_learning_to_predict_together_or_alone\DataCleaning\clean"
OUT_DIR   = r"C:\Users\CR58XM\Documents\GitHub\AAU_learning_to_predict_together_or_alone\Outputs"
MODEL_DIR = r"C:\Users\CR58XM\Documents\GitHub\AAU_learning_to_predict_together_or_alone\Models"

countries = ["Germany", "Ireland", "Portugal"]
days = ["day1", "day2", "day3", "day4", "day5"]

features = [
    "temperature_2m",
    "relative_humidity_2m",
    "wind_speed_10m",
    "precipitation",
    "direct_radiation",
]

PRED_LEN = 96
TAIL_TRAIN_PREDICT = 10000

# Choose model: "base" or "finetuned"
USE_MODEL = "finetuned"  # "base" or "finetuned"

# ------------------------------------------------------------
# Load day cutoffs
# ------------------------------------------------------------
with open(DAYS_JSON, "r") as f:
    dataset_days = json.load(f)

os.makedirs(OUT_DIR, exist_ok=True)

# ------------------------------------------------------------
# Load base pipeline once (only used if USE_MODEL="base")
# ------------------------------------------------------------
base_pipeline = None
if USE_MODEL.lower() == "base":
    base_pipeline = Chronos2Pipeline.from_pretrained("amazon/chronos-2", device_map="cuda")

# Cache finetuned pipelines per country so we don't reload repeatedly
finetuned_cache: dict[str, Chronos2Pipeline] = {}

def get_pipeline_for_country(country: str) -> Chronos2Pipeline:
    """
    Returns the selected pipeline (base or finetuned) for the given country.
    Finetuned models are loaded once and cached.
    """
    if USE_MODEL.lower() == "base":
        return base_pipeline

    if country not in finetuned_cache:
        model_dir = os.path.join(MODEL_DIR, f"Chronos2_LORA_{country}")  # adjust if your naming differs
        finetuned_cache[country] = Chronos2Pipeline.from_pretrained(model_dir, device_map="cuda")
    return finetuned_cache[country]

# ------------------------------------------------------------
# Predict for all countries & days
# ------------------------------------------------------------
for country in countries:
    print("\n==============================")
    print("Country:", country)

    # Load dataset once per country
    data_path = os.path.join(DATA_DIR, f"dataset_{country.capitalize()}.csv")
    df = pd.read_csv(data_path, index_col="timestamp", parse_dates=True).sort_index()
    households = [col for col in df.columns if col not in features]

    # Load correct pipeline for this country
    pipeline = get_pipeline_for_country(country)

    for day in days:
        cutoff = pd.to_datetime(dataset_days[country][day])
        print("  Day:", day, "| cutoff:", cutoff)

        predictions_df_all_households = None

        for h in households:
            s_train = df.loc[df.index < cutoff, h]
            X_train = df.loc[s_train.index, features].copy().ffill().bfill()

            df_household_train = pd.concat([s_train.rename(h), X_train], axis=1)
            df_household_train_last = df_household_train.tail(TAIL_TRAIN_PREDICT).reset_index()
            df_household_train_last["id"] = h

            pred_df = pipeline.predict_df(
                df_household_train_last,
                prediction_length=PRED_LEN,
                id_column="id",
                timestamp_column="timestamp",
                target=h,
            )

            if predictions_df_all_households is None:
                pred_index = pred_df["timestamp"] if "timestamp" in pred_df.columns else pred_df.index
                predictions_df_all_households = pd.DataFrame(index=pred_index)

            predictions_df_all_households[h] = pred_df["predictions"].to_numpy()

        # Save predictions for this (country, day)
        out_path = os.path.join(
            OUT_DIR,
            f"Chronos2_Univar_{USE_MODEL}_pred_{day}_{country.capitalize()}.csv"
        )
        predictions_df_all_households.to_csv(out_path, index=True)
        print("    Saved:", out_path)


Country: Germany
  Day: day1 | cutoff: 2019-12-29 00:00:00
    Saved: C:\Users\CR58XM\Documents\GitHub\AAU_learning_to_predict_together_or_alone\Outputs\Chronos2_finetuned_pred_day1_Germany.csv
  Day: day2 | cutoff: 2019-07-04 00:00:00
    Saved: C:\Users\CR58XM\Documents\GitHub\AAU_learning_to_predict_together_or_alone\Outputs\Chronos2_finetuned_pred_day2_Germany.csv
  Day: day3 | cutoff: 2019-05-18 00:00:00
    Saved: C:\Users\CR58XM\Documents\GitHub\AAU_learning_to_predict_together_or_alone\Outputs\Chronos2_finetuned_pred_day3_Germany.csv
  Day: day4 | cutoff: 2019-12-28 00:00:00
    Saved: C:\Users\CR58XM\Documents\GitHub\AAU_learning_to_predict_together_or_alone\Outputs\Chronos2_finetuned_pred_day4_Germany.csv
  Day: day5 | cutoff: 2019-06-24 00:00:00
    Saved: C:\Users\CR58XM\Documents\GitHub\AAU_learning_to_predict_together_or_alone\Outputs\Chronos2_finetuned_pred_day5_Germany.csv

Country: Ireland
  Day: day1 | cutoff: 2020-12-25 00:00:00
    Saved: C:\Users\CR58XM\Documents\